# Data vetting, cleaning and EDA

MSDS 692 practicum, Regis University. Michael Elliott.

Everything the models use passes through here first. This notebook runs on the full private
files (every row, from the first date to the last day pulled) and keeps the outputs, so the
cleaning decisions can be checked without the data. No search phrases, coordinates or business
names are printed anywhere; businesses are sector labels. If the private files are missing it
runs on the fake sample data and says so.

Sections: 1 what came in · 2 Search Console over time · 3 the rank-tracker problem and the rule ·
4 Maps scans: duplicates, broken scans, triage · 5 distributions · 6 the older scans I left out ·
7 GA4 · 8 the dance studio's 14-month climb (slide 5) · 9 what each cleaning step cost

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import reach, synthetic_control as sc, sample_data
P = ROOT / "data" / "private"
REAL = (P / "gsc_daily.parquet").exists() and (P / "geogrids.parquet").exists()
if REAL:
    GSC, GEO = P / "gsc_daily.parquet", P / "geogrids.parquet"
    THREE = ["Specialty Barbershop", "Latin Dance Studio", "Integrative Health Clinic"]
    CFG = json.loads((P / "clients.json").read_text()) if (P / "clients.json").exists() else {}
    print("running on the real files (not in the repo)")
else:
    if not (ROOT / "data/sample/geogrids.parquet").exists(): sample_data.make()
    GSC, GEO = ROOT / "data/sample/gsc_daily.parquet", ROOT / "data/sample/geogrids.parquet"
    THREE = ["Sample barbershop", "Sample dance studio", "Sample clinic"]; CFG = {}
    print("running on the fake sample data: the numbers below mean nothing")
def have(name): return REAL and (P / name).exists()
def pct(a, b): return f"{100*a/b:.1f}%" if b else ""
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

running on the real files (not in the repo)


## 1. What came in

In [2]:
g = pd.read_parquet(GSC); g["date"] = pd.to_datetime(g["date"])
geo = pd.read_parquet(GEO); geo["scan_date"] = pd.to_datetime(geo["scan_date"], format="ISO8601", utc=True).dt.tz_localize(None).dt.normalize()
rows = [["Search Console rows (date x device x phrase)", len(g), g.date.min().date(), g.date.max().date(), g.client.nunique()],
        ["Maps scan grid points", len(geo), geo.scan_date.min().date(), geo.scan_date.max().date(), geo.business_name.nunique()],
        ["Maps scans (business x phrase x date)", geo.heatmap_id.nunique(), geo.scan_date.min().date(), geo.scan_date.max().date(), geo.business_name.nunique()]]
if have("ga4.parquet"):
    a4 = pd.read_parquet(P / "ga4.parquet"); a4["date"] = pd.to_datetime(a4["date"])
    rows.append(["GA4 rows (date x channel)", len(a4), a4.date.min().date(), a4.date.max().date(), a4.client.nunique()])
if have("historical_geogrids.parquet"):
    h = pd.read_parquet(P / "historical_geogrids.parquet"); h["scan_date"] = pd.to_datetime(h["scan_date"], format="ISO8601", utc=True, errors="coerce").dt.tz_localize(None)
    rows.append(["Older scans from a different tool (grid points)", len(h), h.scan_date.min().date(), h.scan_date.max().date(), h.business_name.nunique()])
came_in = pd.DataFrame(rows, columns=["source", "rows", "first", "last", "businesses"]); came_in

,source,rows,first,last,businesses
0,Search Console rows (date x device x phrase),26756,2025-08-25,2026-10-18,6
1,Maps scan grid points,32714,2026-06-16,2026-10-14,8
2,Maps scans (business x phrase x date),206,2026-06-16,2026-10-14,8
3,GA4 rows (date x channel),1988,2026-03-24,2026-10-18,6
4,Older scans from a different tool (grid points),13447,2026-06-29,2026-08-06,2


In [3]:
print("latest Search Console day per client:"); display(g.groupby("client").date.max().dt.date.to_frame("last day").T)
print("Search Console rows per client per month:")
g.assign(m=g.date.dt.to_period("M")).pivot_table(index="m", columns="client", values="impressions", aggfunc="size").fillna(0).astype(int)

latest Search Console day per client:


client,Dental Practice,Fractional-CTO Consultancy,Insulation Contractor,Integrative Health Clinic,Latin Dance Studio,Specialty Barbershop
last day,2026-10-18,2026-10-18,2026-10-18,2026-10-18,2026-10-18,2026-10-17


Search Console rows per client per month:


client,Dental Practice,Fractional-CTO Consultancy,Insulation Contractor,Integrative Health Clinic,Latin Dance Studio,Specialty Barbershop
m,,,,,,
2025-08,0,0,0,0,55,0
2025-09,0,0,0,0,77,0
2025-10,0,0,0,0,145,0
2025-11,0,0,0,0,160,0
2025-12,0,0,0,0,281,0
2026-01,0,0,0,0,679,0
2026-02,0,0,0,0,919,0
2026-03,0,0,26,0,837,0
2026-04,0,0,115,0,739,0


## 2. Search Console over time

Impressions per day, all rows, before any cleaning. 7-day average, one line per client.

In [4]:
daily_all = g.pivot_table(index="date", columns="client", values="impressions", aggfunc="sum").asfreq("D").fillna(0)
fig, ax = plt.subplots(figsize=(12, 4.5))
for c in daily_all.columns:
    ax.plot(daily_all.index, daily_all[c].rolling(7, min_periods=1).mean(), lw=1.6, label=c)
ax.set_ylabel("impressions per day, 7-day average"); ax.set_yscale("log"); ax.legend(frameon=False, ncol=3, fontsize=8); ax.set_title("Raw Search Console impressions, every client")
plt.show()
print("device split of impressions, all rows:"); g.pivot_table(index="client", columns="device", values="impressions", aggfunc="sum").fillna(0).astype(int)

device split of impressions, all rows:


device,DESKTOP,MOBILE,TABLET
client,,,
Dental Practice,9798,5612,68
Fractional-CTO Consultancy,857,620,9
Insulation Contractor,8117,976,15
Integrative Health Clinic,13944,864,16
Latin Dance Studio,9394,9151,134
Specialty Barbershop,1616,978,2


## 3. The rank-tracker problem, and the rule

The rank tracker used for the Maps scans searches Google from a desktop emulator every day and never clicks.
In Search Console that shows up as desktop rows with zero clicks, a pile of impressions on one
phrase, and a position near 1. The chart splits desktop from mobile for the clinic, where it was
worst. Then the position histogram of desktop zero-click rows shows why the cutoff is 2.5 and
not exactly 1.0: the tracker's local-pack hits land at 1.1 to 1.9 too.

In [5]:
clinic = THREE[2]
d = g[g.client == clinic].pivot_table(index="date", columns="device", values="impressions", aggfunc="sum").asfreq("D").fillna(0)
fig, ax = plt.subplots(figsize=(12, 4))
for dev in d.columns: ax.plot(d.index, d[dev].rolling(7, min_periods=1).mean(), lw=1.8, label=dev)
ax.set_title(f"{clinic}: impressions per day by device (7-day average)"); ax.legend(frameon=False); plt.show()
z = g[(g.device == "DESKTOP") & (g.clicks == 0) & (g.impressions >= 20)]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(z.position.clip(upper=15), bins=np.arange(0.5, 16, 0.5), color="#2563eb")
ax.axvline(2.5, color="#ea580c", ls="--"); ax.set_xlabel("average position of desktop, zero-click rows with 20+ impressions"); ax.set_ylabel("rows")
ax.set_title("where the tracker rows sit: a cutoff at exactly 1.0 would miss the 1.1 to 1.9 hits"); plt.show()
g["is_scan"] = sc.scan_mask(g)
rm = g.groupby("client").apply(lambda x: pd.Series({"impressions": x.impressions.sum(), "removed": x.loc[x.is_scan, "impressions"].sum()}))
rm["share removed"] = (100 * rm.removed / rm.impressions).round(1); rm.astype({"impressions": int, "removed": int})

,impressions,removed,share removed
client,,,
Dental Practice,15478,0,0.0
Fractional-CTO Consultancy,1486,39,2.6
Insulation Contractor,9108,1021,11.2
Integrative Health Clinic,14824,10930,73.7
Latin Dance Studio,18679,1423,7.6
Specialty Barbershop,2596,1367,52.7


In [6]:
print("sanity check: before the project's scanning started (first scan in the file:", geo.scan_date.min().date(), ") the rule should remove little. What it does remove there is the rule's false-positive cost: real desktop rows that happen to look like a tracker.")
first = geo.scan_date.min()
before = g[g.date < first]; after = g[g.date >= first]
pd.DataFrame({"period": ["before the first scan", "after"], "impressions": [before.impressions.sum(), after.impressions.sum()],
              "removed by the rule": [before.loc[before.is_scan, "impressions"].sum(), after.loc[after.is_scan, "impressions"].sum()]})

sanity check: before the project's scanning started (first scan in the file: 2026-06-16 ) the rule should remove little. What it does remove there is the rule's false-positive cost: real desktop rows that happen to look like a tracker.


,period,impressions,removed by the rule
0,before the first scan,9670,385
1,after,52501,14395


## 4. Maps scans: duplicates, broken scans, triage

A scan is one business, one phrase, one date, a grid of points. Three kinds of trouble showed up:
scans run twice on the same day, scans whose points came back without usable coordinates (the
grid half-width computes to thousands of miles), and scans that never ran against a real result
set (no rank at any point). The Week 4 triage file carries the API status fields for the scans
that existed then; for the newer scans the same checks are applied here from the points.

In [7]:
DEG = 69.055
def halfwidth(s):
    clat, clng = s.center_lat.iloc[0], s.center_lng.iloc[0]
    return float(max((s.lat - clat).abs().max() * DEG, (s.lng - clng).abs().max() * DEG * np.cos(np.radians(clat))))
scans = (geo.groupby("heatmap_id").apply(lambda s: pd.Series({
    "business": s.business_name.iloc[0], "date": s.scan_date.iloc[0].date(), "points": len(s),
    "ranked": int(s["rank"].notna().sum()), "top3": int(s.is_top3.sum()),
    "grid_mi": float(s.distance_miles.max()) if s.distance_miles.notna().any() else np.nan,
    "half_mi": halfwidth(s) if s[["lat", "lng"]].notna().all(axis=None) else np.nan,
    "phrase_key": hash((s.business_name.iloc[0], s.keyword.iloc[0], s.scan_date.iloc[0].date()))})).reset_index())
scans["broken coordinates"] = scans.half_mi > 1000
scans["no distances"] = scans.grid_mi.isna()
scans["no ranks at all"] = scans.ranked == 0
scans["grid too small (< 0.5 mi)"] = scans.grid_mi < reach.MIN_GRID_MI
scans["grid too big (> 60 mi)"] = scans.grid_mi > reach.MAX_GRID_MI
scans["duplicate (same business, phrase, day)"] = scans.duplicated("phrase_key", keep=False)
flags = ["broken coordinates", "no distances", "no ranks at all", "grid too small (< 0.5 mi)", "grid too big (> 60 mi)", "duplicate (same business, phrase, day)"]
print(f"{len(scans)} scans, {scans.business.nunique()} businesses")
print("scans per business per month:"); display(scans.assign(m=pd.to_datetime(scans.date).dt.to_period("M")).pivot_table(index="business", columns="m", values="heatmap_id", aggfunc="size").fillna(0).astype(int))
print("problems found (a scan can have more than one):"); display(scans[flags].sum().to_frame("scans"))
usable = scans[~(scans["broken coordinates"] | scans["no distances"] | scans["no ranks at all"] | scans["grid too small (< 0.5 mi)"] | scans["grid too big (> 60 mi)"])]
dup_kept = usable.duplicated("phrase_key", keep=False).sum()
print(f"scans kept for modelling: {len(usable)} of {len(scans)}  ({pct(len(usable), len(scans))})")
print(f"most 'duplicates' are a broken-coordinate copy of a good scan (the same scan ingested twice); among the kept scans {dup_kept} still share a business, phrase and day. Those are real repeat scans and stay in; the per-business medians are read with that in mind.")

206 scans, 8 businesses
scans per business per month:


m,2026-06,2026-07,2026-08,2026-09,2026-10
business,,,,,
Competitor / Peer,3,14,0,0,3
Dental Practice,0,0,0,18,0
Direct Primary Care Clinic,0,0,2,0,0
Fractional-CTO Consultancy,0,20,0,0,0
Insulation Contractor,5,0,0,8,7
Integrative Health Clinic,0,0,6,51,10
Latin Dance Studio,3,0,2,13,9
Specialty Barbershop,2,2,2,6,20


problems found (a scan can have more than one):


,scans
broken coordinates,63
no distances,63
no ranks at all,0
grid too small (< 0.5 mi),4
grid too big (> 60 mi),3
"duplicate (same business, phrase, day)",151


scans kept for modelling: 136 of 206  (66.0%)
most 'duplicates' are a broken-coordinate copy of a good scan (the same scan ingested twice); among the kept scans 95 still share a business, phrase and day. Those are real repeat scans and stay in; the per-business medians are read with that in mind.


In [8]:
if have("geogrids_status.parquet"):
    st = pd.read_parquet(P / "geogrids_status.parquet")
    print("API status fields for the scans in the older full pull:"); display(st.status.value_counts().to_frame("scans"))
    print("points processed per scan (should equal points):"); display((st.processed == st.points).value_counts().to_frame("scans"))
if have("triage.parquet"):
    tri = pd.read_parquet(P / "triage.parquet")
    print(f"Week 4 triage, {len(tri)} scans as of September 20: usable {int(tri.usable.sum())}, not usable {int((~tri.usable).sum())}")
    display(tri.reason.value_counts().to_frame("scans"))

API status fields for the scans in the older full pull:


,scans
status,
Complete,142
In Progress,1


points processed per scan (should equal points):


,scans
True,142
False,1


Week 4 triage, 143 scans as of September 20: usable 114, not usable 29


,scans
reason,
decay fittable,46
"floor: never top-3, radius is zero inside this grid",42
too few top-3 points to fit a decay,18
grid radius 0.02 mi is too small to show any decay,10
every point came back with no rank at all - scan did not run against a real result set,8
"ceiling: top-3 at every point, radius exceeds the grid",8
grid radius 0.03 mi is too small to show any decay,4
grid radius 0.42 mi is too small to show any decay,2
grid radius 105 mi spans other metros,2


## 5. Distributions

In [9]:
u = geo[geo.heatmap_id.isin(usable.heatmap_id)]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
axes[0].hist(u["rank"].dropna().clip(upper=21), bins=np.arange(0.5, 22, 1), color="#2563eb"); axes[0].set_title("rank at a grid point (21 = beyond 20)"); axes[0].set_xlabel("rank")
axes[1].hist(usable.top3 / usable.points, bins=20, color="#2563eb"); axes[1].set_title("share of points in the top 3, per scan"); axes[1].set_xlabel("share")
axes[2].hist(usable.grid_mi.clip(upper=40), bins=30, color="#2563eb"); axes[2].set_title("farthest grid point, per scan"); axes[2].set_xlabel("miles")
plt.tight_layout(); plt.show()
print("points per scan:", usable.points.value_counts().to_dict())
print("missing values per column, grid points:"); display(geo.isna().sum().to_frame("missing").T)

points per scan: {169: 85, 137: 38, 170: 11, 81: 1, 49: 1}
missing values per column, grid points:


,heatmap_id,business_name,place_id,keyword,search_type,grid_size,scan_date,center_lat,center_lng,point_index,lat,lng,distance_miles,rank,is_top3,average_position,top3_percentage,grid_radius,pin_lat,pin_lng
missing,0,0,11,0,0,0,0,11,11,11,11,11,10143,0,0,11,11,25587,25587,25587


## 6. The older scans I left out

Before mid-June the scans came from a different tool with much larger grids and a different point layout. They are counted here and not used: the radii would not be comparable.

In [10]:
if have("historical_geogrids.parquet"):
    hh = h.copy()
    print(f"{len(hh):,} points, {hh.business_name.nunique()} businesses, {hh.scan_date.min().date()} to {hh.scan_date.max().date()}")
    display(hh.groupby("business_name").agg(points=("rank", "size"), scans=("scan_date", "nunique"), grid_sizes=("grid_size", lambda s: sorted(s.dropna().unique().tolist())), radius_values=("radius", lambda s: sorted(s.dropna().unique().tolist())[:6])))
else:
    print("not available in the sample data")

13,447 points, 2 businesses, 2026-06-29 to 2026-08-06


,points,scans,grid_sizes,radius_values
business_name,,,,
Insulation Contractor,7164,7,"[5x5, 9x9]",[5.0mi]
Integrative Health Clinic,6283,2,"[21x21, 9x9]","[20.0mi, 5.0mi]"


## 7. GA4 sessions

Organic sessions per day per client, the only GA4 series used (as a hint next to the Search Console model).

In [11]:
if have("ga4.parquet"):
    org = a4[a4.channel.str.contains("Organic Search", case=False, na=False)].pivot_table(index="date", columns="client", values="sessions", aggfunc="sum").asfreq("D").fillna(0)
    fig, ax = plt.subplots(figsize=(12, 4))
    for c in org.columns: ax.plot(org.index, org[c].rolling(7, min_periods=1).mean(), lw=1.6, label=c)
    ax.set_title("GA4 organic sessions per day, 7-day average"); ax.legend(frameon=False, ncol=3, fontsize=8); plt.show()
    print("latest GA4 day per client:"); display(a4.groupby("client").date.max().dt.date.to_frame("last day").T)
else:
    print("not available in the sample data")

latest GA4 day per client:


client,Dental Practice,Fractional-CTO Consultancy,Insulation Contractor,Integrative Health Clinic,Latin Dance Studio,Specialty Barbershop
last day,2026-10-17,2026-10-04,2026-10-17,2026-10-17,2026-10-18,2026-10-17


## 8. The dance studio's 14-month climb (slide 5)

Monthly impression-weighted average web position of the dance studio for its class-related
searches, people only (tracker rows removed) and brand searches removed (the brand pattern is read
from the private file and never printed). This is the right-hand chart on slide 5.

In [12]:
ds = THREE[1]
x = g[(g.client == ds) & ~g.is_scan].copy()
pat = CFG.get("brand_patterns", {}).get(ds)
if pat: x = x[~x["query"].str.contains(pat, case=False, regex=True, na=False)]
x = x[x["query"].str.contains("class", case=False, na=False)]
if len(x):
    m = x.assign(m=x.date.dt.to_period("M")).groupby("m").apply(lambda d: pd.Series({"impressions": d.impressions.sum(), "position": np.average(d.position, weights=d.impressions)}))
    m["position"] = m.position.round(1)
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.plot(m.index.to_timestamp(), m.position, marker="o", color="#2563eb", lw=2); ax.invert_yaxis(); ax.axhspan(0, 10.5, color="#2563eb", alpha=0.06)
    ax.set_ylabel("average web position (lower is better)"); ax.set_title(f"{ds}: class-related searches, monthly, people only"); plt.show()
    display(m.T)
else:
    print("no class-related rows in this data")

m,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09,2026-10
impressions,3.0,6.0,36.0,34.0,73.0,265.0,449.0,335.0,271.0,249.0,269.0,192.0,187.0,178.0,137.0
position,64.3,28.5,9.6,10.1,8.1,11.4,8.7,8.3,9.6,9.1,8.4,8.8,11.9,9.3,13.6


## 9. What each cleaning step cost

In [13]:
cost = [["Search Console: tracker rows removed (rule in section 3)", int(g.loc[g.is_scan, "impressions"].sum()), "impressions", pct(g.loc[g.is_scan, "impressions"].sum(), g.impressions.sum())],
        ["Maps scans: broken coordinates / no distances", int((scans["broken coordinates"] | scans["no distances"]).sum()), "scans", pct((scans["broken coordinates"] | scans["no distances"]).sum(), len(scans))],
        ["Maps scans: no rank at any point", int(scans["no ranks at all"].sum()), "scans", pct(scans["no ranks at all"].sum(), len(scans))],
        ["Maps scans: grid under 0.5 or over 60 miles", int((scans["grid too small (< 0.5 mi)"] | scans["grid too big (> 60 mi)"]).sum()), "scans", pct((scans["grid too small (< 0.5 mi)"] | scans["grid too big (> 60 mi)"]).sum(), len(scans))],
        ["Older scans from the other tool, not used", int(len(h)) if have("historical_geogrids.parquet") else 0, "grid points", ""]]
pd.DataFrame(cost, columns=["step", "removed", "unit", "share"])

,step,removed,unit,share
0,Search Console: tracker rows removed (rule in ...,14780,impressions,23.8%
1,Maps scans: broken coordinates / no distances,63,scans,30.6%
2,Maps scans: no rank at any point,0,scans,0.0%
3,Maps scans: grid under 0.5 or over 60 miles,7,scans,3.4%
4,"Older scans from the other tool, not used",13447,grid points,
